# Homework 1: Single-Cell RNA-Seq Analysis with Seurat — ✅ COMPLETED VERSION
## Computational Biology at Berkeley (CBAB) Bootcamp

> **This is the answer key.** All blanks are filled in and annotated with explanations.

### Running this notebook

| Platform | How to open |
|----------|-------------|
| **VS Code** | Install the *Jupyter* extension → open this `.ipynb` → select the **R** kernel |
| **JupyterLab / Jupyter Notebook** | `jupyter lab` or `jupyter notebook` in your terminal → navigate to this file |
| **Google Colab** | File → Upload notebook → Runtime → Change runtime type → **R** |

> ⚙️ First time? See the **README** for a one-time local setup guide.

### Dataset
**PBMC 3k** — 2,700 Peripheral Blood Mononuclear Cells (10x Genomics)
Reference tutorial: [Seurat PBMC3k vignette](https://satijalab.org/seurat/articles/pbmc3k_tutorial)

---

In [ ]:
# ── ENVIRONMENT SETUP ────────────────────────────────────────────────────────
# LOCAL (VS Code / Jupyter):
#   1. Install R ≥ 4.1  →  https://cran.r-project.org
#   2. pip install jupyterlab
#   3. In an R console: install.packages("IRkernel"); IRkernel::installspec()
#   4. Restart VS Code / Jupyter — select the "R" kernel.
#
# GOOGLE COLAB:
#   Runtime → Change runtime type → R  (then run this cell first)
# ─────────────────────────────────────────────────────────────────────────────

# On Linux/Colab: use Posit Package Manager pre-built binaries.
# This avoids compiling Seurat's C++ dependencies from source,
# cutting install time from ~15 minutes down to ~2 minutes.
if (Sys.info()[["sysname"]] == "Linux") {
  options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
}

if (!requireNamespace("Seurat",    quietly = TRUE)) install.packages("Seurat")
if (!requireNamespace("dplyr",     quietly = TRUE)) install.packages("dplyr")
if (!requireNamespace("patchwork", quietly = TRUE)) install.packages("patchwork")

cat("Seurat version:", as.character(packageVersion("Seurat")), "\n")

In [ ]:
# Download and unpack the PBMC 3k dataset automatically
if (!dir.exists("filtered_gene_bc_matrices")) {
  url  <- "https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"
  dest <- "pbmc3k_filtered_gene_bc_matrices.tar.gz"
  download.file(url, destfile = dest, mode = "wb")
  untar(dest)
  message("Download complete and data extracted.")
} else {
  message("Data folder already exists — skipping download.")
}
list.dirs("filtered_gene_bc_matrices", recursive = FALSE)

## Section 1: Loading the Data

`Read10X()` loads the raw UMI count matrix from the 10x Genomics output directory.
`CreateSeuratObject()` wraps it into Seurat's core data structure.

Key parameters for `CreateSeuratObject()`:

| Parameter | Meaning |
|-----------|---------|
| `counts` | the raw count matrix |
| `project` | a label for this dataset |
| `min.cells` | keep a gene only if it is detected in **at least** this many cells (removes ultra-rare/noisy genes) |
| `min.features` | keep a cell only if it expresses **at least** this many genes (removes empty droplets) |

In [ ]:
library(Seurat)
library(dplyr)
library(patchwork)

# Load the PBMC dataset
pbmc.data <- Read10X(data.dir = "filtered_gene_bc_matrices/hg19/")

# Create the Seurat object.
# min.cells = 3  → removes genes detected in fewer than 3 cells (reduces noise)
# min.features = 200 → removes droplets with <200 genes (likely empty droplets)
pbmc <- CreateSeuratObject(counts = pbmc.data, project = "pbmc3k",
                           min.cells = 3, min.features = 200)
pbmc

## Section 2: Quality Control (QC)

Not every "cell" in the dataset is a real, intact cell.  Common technical artifacts include:

- **Empty droplets** — the droplet captured no cell; shows very few genes (`nFeature_RNA` low)
- **Doublets** — two cells merged into one droplet; shows unusually many genes/reads
- **Damaged / dying cells** — the cytoplasm leaked out but mitochondria are still intact,
  so a high fraction of reads map to mitochondrial genes (`percent.mt` high)

We use **three QC metrics** to identify and remove these:

| Metric | Good range | Problem if low | Problem if high |
|--------|-----------|----------------|-----------------|
| `nFeature_RNA` | 200 – 2500 | empty droplet | doublet |
| `nCount_RNA` | moderate | empty droplet | doublet |
| `percent.mt` | < 5 % | — | dying cell |

In [ ]:
# Calculate the fraction of reads that map to mitochondrial genes.
# The regex "^MT-" matches any gene whose name starts with "MT-".
pbmc[["percent.mt"]] <- PercentageFeatureSet(pbmc, pattern = "^MT-")

### Visualising the QC Distributions

The violin plots below show the three metrics across all cells.
Examine each plot: where do you see outliers that should be removed?

In [ ]:
# Violin plots of QC metrics (given — just run this cell)
VlnPlot(pbmc, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), ncol = 3)

### Feature Scatter & Filtering

Scatter plots reveal how the metrics co-vary, helping you confirm that your chosen
thresholds capture real outliers and not just normal variation.

In [ ]:
# Scatter plots
plot1 <- FeatureScatter(pbmc, feature1 = "nCount_RNA", feature2 = "percent.mt")
plot2 <- FeatureScatter(pbmc, feature1 = "nCount_RNA", feature2 = "nFeature_RNA")
plot1 + plot2

# Filter: keep cells with 200–2500 genes AND < 5 % mitochondrial reads
pbmc <- subset(pbmc, subset = nFeature_RNA > 200 & nFeature_RNA < 2500 & percent.mt < 5)

In [ ]:
# Violin plots AFTER filtering — verify that outliers are gone
VlnPlot(pbmc, features = c("nFeature_RNA", "nCount_RNA", "percent.mt"), ncol = 3)

## Section 3: Normalization

Even after QC, cells have different total read counts due to technical variation
in sequencing depth. **Normalization** removes this bias.

Seurat's default `"LogNormalize"` method:
1. Divides each gene's count by the cell's total counts
2. Multiplies by a scale factor (default 10,000 → "counts per 10 k", CP10K)
3. Log-transforms: `log1p(CP10K)` — compresses extreme values and makes the distribution more symmetric

In [ ]:
# Normalize: LogNormalize + scale factor 10,000 (Seurat defaults)
pbmc <- NormalizeData(pbmc)

## Section 4: Feature Selection — Highly Variable Genes (HVGs)

Most genes barely change across cell types ("housekeeping" genes).
To focus downstream analysis on meaningful biology, we select the **top 2,000 highly variable genes (HVGs)** —
genes that vary more than expected given their average expression level.

`FindVariableFeatures()` with `selection.method = "vst"` (variance-stabilizing transformation):
- Models the mean–variance relationship across genes
- Flags genes where observed variance exceeds the model expectation
- Returns a ranked list with the most variable genes at the top

In [ ]:
# Identify the top 2000 highly variable genes using variance-stabilizing transformation
pbmc <- FindVariableFeatures(pbmc, selection.method = "vst", nfeatures = 2000)

# Identify the 10 most highly variable genes
top10 <- head(VariableFeatures(pbmc), 10)

# Plot variable features with and without labels
plot1 <- VariableFeaturePlot(pbmc)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot1
plot2

## Section 5: Scaling the Data

Before PCA, we **scale** each gene so that it has:
- **Mean = 0** (centering)
- **Variance = 1** (unit variance)

Without scaling, highly expressed genes would dominate PCA simply because they
have larger absolute values — not because they're biologically more important.

We scale across **all genes** (not just HVGs) so that heatmaps later display expression correctly.

In [ ]:
# Scale all genes: center (mean=0) and scale (variance=1) each gene across cells
all.genes <- rownames(pbmc)
pbmc <- ScaleData(pbmc, features = all.genes)

## Section 6: Principal Component Analysis (PCA)

PCA compresses thousands of gene-expression dimensions into a small number of
**principal components (PCs)** that capture the major axes of variation.

- Each PC is a weighted combination of genes (*gene loadings*)
- PC1 captures the most variance, PC2 the second-most, and so on
- Biologically, one PC might separate lymphocytes from monocytes;
  another might separate naive from activated T cells

We run PCA on the **highly variable genes** identified in the previous step.

In [ ]:
# Run PCA using the top 2000 highly variable genes
pbmc <- RunPCA(pbmc, features = VariableFeatures(object = pbmc))

### Interpreting PCA Results

**Gene loadings** tell us which genes drive each PC:
- Large positive loading → gene is highly expressed in cells on the positive end of the PC
- Large negative loading → gene is highly expressed in cells on the negative end

The plots below help us understand the biological meaning of each PC.

In [ ]:
# Top genes driving PC1 and PC2 (given — just run this cell)
VizDimLoadings(pbmc, dims = 1:2, reduction = "pca")

# Scatter plot of all cells in PC1 vs PC2 space
DimPlot(pbmc, reduction = "pca") + NoLegend()

### PCA Heatmaps

Each heatmap shows the top genes driving one PC, across 500 cells ordered by their PC score.
**Purple = low expression · Yellow = high expression**
A clear purple-to-yellow split indicates a strong, biologically meaningful component.

In [ ]:
# Heatmap for PC1
DimHeatmap(pbmc, dims = 1, cells = 500, balanced = TRUE)

# Heatmaps for PCs 1–15 (helps assess which PCs are informative)
DimHeatmap(pbmc, dims = 1:15, cells = 500, balanced = TRUE)

## Section 7: Determining Dataset Dimensionality

How many PCs should we use for clustering?

- **Too few** → miss real biological variation
- **Too many** → include noise, slow computation

The **elbow plot** ranks PCs by the percentage of variance explained.
The "elbow" — where the curve flattens — marks the transition from
informative PCs (left) to mostly noise (right).

> For this dataset you should observe an elbow around **PC 9–10**.

In [ ]:
# Elbow plot — variance explained by each PC
ElbowPlot(pbmc)

# The curve flattens around PC 9–10, meaning the first 10 PCs capture most
# of the true biological signal. We will use dims = 1:10 for all downstream steps.

## Section 8: Finding Neighbors & Clustering

Seurat uses a **graph-based clustering** approach:

1. **`FindNeighbors()`** — builds a K-nearest-neighbor (KNN) graph in PC space.
   Each cell is connected to its *k* most similar neighbors.
2. **`FindClusters()`** — runs the Louvain community-detection algorithm
   to identify densely connected groups (clusters).

The `resolution` parameter controls granularity:

| Resolution | Effect |
|-----------|--------|
| 0.2 – 0.4 | Fewer, larger clusters |
| 0.5 – 0.8 | Moderate (good starting point) |
| 1.0 – 1.2 | Many, finer-grained clusters |

In [ ]:
# Build KNN graph using the first 10 PCs
pbmc <- FindNeighbors(pbmc, dims = 1:10)

# Cluster with Louvain algorithm, resolution = 0.5
pbmc <- FindClusters(pbmc, resolution = 0.5)

# Check cluster IDs for the first 5 cells
head(Idents(pbmc), 5)

## Section 9: Dimensionality Reduction for Visualization

Clusters exist in ~10-dimensional PC space — hard to visualise!
**t-SNE** and **UMAP** project cells into 2D while preserving local neighborhood structure,
so similar cells end up close together in the plot.

| Method | Pros | Cons |
|--------|------|------|
| t-SNE | Clear cluster separation | Slow; global distances not meaningful |
| UMAP | Fast; some global structure preserved | Newer, less studied statistically |

> ⚠️ **Important:** cluster distances in t-SNE/UMAP are **not** directly comparable.
> Use these plots for visualization only, not to conclude how "different" clusters are.

In [ ]:
# t-SNE — classic method, 10 PCs
pbmc <- RunTSNE(pbmc, dims = 1:10)
DimPlot(pbmc, reduction = "tsne")

# UMAP — faster, preferred for large datasets, 10 PCs
pbmc <- RunUMAP(pbmc, dims = 1:10)
DimPlot(pbmc, reduction = "umap")

## Section 10: Finding Cluster Biomarkers

To understand what each cluster represents biologically, we find **marker genes** —
genes significantly more expressed in one cluster compared to all others.

`FindMarkers()` runs a Wilcoxon rank-sum test (default) and returns:

| Column | Meaning |
|--------|---------|
| `p_val_adj` | Adjusted p-value (use this for significance) |
| `avg_log2FC` | Log2 fold-change (positive = higher in this cluster) |
| `pct.1` | Fraction of cells in **this** cluster expressing the gene |
| `pct.2` | Fraction in **all other** cells expressing the gene |

In [ ]:
# Example: markers for cluster 2 (GIVEN — study this pattern, then do Q11)
cluster2.markers <- FindMarkers(pbmc, ident.1 = 2)
head(cluster2.markers, n = 5)

In [ ]:
# Markers for cluster 3 (B cells — expect MS4A1, CD79A)
cluster3.markers <- FindMarkers(pbmc, ident.1 = 3)
head(cluster3.markers, n = 5)

In [ ]:
# Find positive markers for ALL clusters at once
pbmc.markers <- FindAllMarkers(pbmc, only.pos = TRUE)
pbmc.markers %>%
    group_by(cluster) %>%
    dplyr::filter(avg_log2FC > 1)

In [ ]:
# Violin plot: expression of canonical B-cell markers across all clusters
VlnPlot(pbmc, features = c("MS4A1", "CD79A"))

# Feature plot: overlay 9 key marker genes onto the UMAP embedding
FeaturePlot(pbmc, features = c("MS4A1", "GNLY", "CD3E", "CD14",
                                "FCER1A", "FCGR3A", "LYZ", "PPBP", "CD8A"))

## Section 11: Cell Type Annotation

Now we assign **biological labels** to each cluster based on known marker genes.

Use the feature plots and marker tables above together with this reference:

| Cluster | Key Markers | Cell Type |
|---------|-------------|-----------|
| 0 | IL7R, CCR7 | Naive CD4+ T |
| 1 | CD14, LYZ | CD14+ Monocyte |
| 2 | IL7R, S100A4 | Memory CD4+ T |
| 3 | MS4A1, CD79A | B cell |
| 4 | CD8A | CD8+ T |
| 5 | FCGR3A, MS4A7 | FCGR3A+ Monocyte |
| 6 | GNLY, NKG7 | NK cell |
| 7 | FCER1A, CST3 | Dendritic cell |
| 8 | PPBP | Platelet |

In [ ]:
# Annotate clusters with known immune cell type labels
new.cluster.ids <- c("Naive CD4 T", "CD14+ Mono", "Memory CD4 T", "B",
                     "CD8 T", "FCGR3A+ Mono", "NK", "DC", "Platelet")
names(new.cluster.ids) <- levels(pbmc)
pbmc <- RenameIdents(pbmc, new.cluster.ids)
DimPlot(pbmc, reduction = "umap", label = TRUE, pt.size = 0.5) + NoLegend()